**0. Lembrar de ativar GPUs: T4 no ambiente de execução**

**1. Inserção do Dataset via Kaggle Hub**

In [1]:
import kagglehub
import os
import pandas as pd

# Download do dataset pré-processado do meu repositório público

path = kagglehub.dataset_download("angelosbc/steam-reviews-in-portuguese-pt-br-2021")
print("Pasta do dataset:", path)

# Carrega o CSV filtrado no Pandas
caminho_csv = os.path.join(path, "steam_reviews_brazilian.csv")
df = pd.read_csv(caminho_csv)

# 3. Mostra total de avaliações e alguns dados
print(f"Total de avaliações em português: {len(df):,}")
df.head()

100%|██████████| 96.6M/96.6M [00:05<00:00, 17.9MB/s]

Extracting files...


Pasta do dataset: /root/.cache/kagglehub/datasets/angelosbc/steam-reviews-in-portuguese-pt-br-2021/versions/1
Total de avaliações em português: 918,910


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,29,292030,The Witcher 3: Wild Hunt,85177505,brazilian,Se um dia alguém falar que esse jogo é ruim na...,1611368498,1611368498,True,1,...,True,False,False,76561198844659805,70,4,11115.0,2252.0,11115.0,1.611186e+09
1,30,292030,The Witcher 3: Wild Hunt,85176839,portuguese,bom demais\n,1611367482,1611367482,True,0,...,True,False,False,76561198847379347,4,1,555.0,465.0,555.0,1.611367e+09
2,32,292030,The Witcher 3: Wild Hunt,85176661,brazilian,NaN,1611367193,1611367193,True,0,...,True,False,False,76561198076880796,127,13,875.0,752.0,826.0,1.611370e+09
3,34,292030,The Witcher 3: Wild Hunt,85176249,brazilian,Obra prima!!!,1611366524,1611366524,True,0,...,True,False,False,76561198957873353,32,1,2888.0,1475.0,2888.0,1.611366e+09
4,43,292030,The Witcher 3: Wild Hunt,85173023,brazilian,Jogão da porra.,1611361229,1611361229,True,0,...,True,False,False,76561198141110905,59,4,20193.0,3692.0,20193.0,1.611297e+09


**2. Pré-processamento Textual e Divisão dos Dados**

In [2]:
import re
from sklearn.model_selection import train_test_split

# 1. Remove linhas nulas nas colunas essenciais
print("-> Removendo valores ausentes...")
df = df.dropna(subset=['review', 'recommended'])

# 2. Converte a coluna alvo para inteiro (0 = Negativo, 1 = Positivo)
df['label'] = df['recommended'].astype(int)

# 3. Função de normalização e limpeza sintática do texto
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Converte para minúsculas
    texto = texto.lower()
    # Remove URLs completas
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # Remove tags HTML
    texto = re.sub(r'<.*?>', '', texto)
    # Mantém apenas letras acentuadas e espaços, removendo pontuações/símbolos
    texto = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', ' ', texto)
    # Remove espaços em branco redundantes
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 4. Aplica a limpeza e filtra textos com menos de 4 caracteres
print("-> Aplicando rotina de limpeza no texto...")
df['clean_review'] = df['review'].apply(limpar_texto)
df = df[df['clean_review'].str.len() >= 4]

print(f"Total de registros válidos pós-limpeza: {len(df):,}")

# 5. Divisão estratificada (70% Treino, 15% Validação, 15% Teste)
print("-> Realizando divisão estratificada (70/15/15)...")
X = df['clean_review']
y = df['label']

# Primeiro corte: separa 70% treino e 30% temporário
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Segundo corte: divide os 30% temporários igualmente entre validação e teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Conjunto de Treino:     {len(X_train):,} amostras")
print(f"Conjunto de Validação:  {len(X_val):,} amostras")
print(f"Conjunto de Teste:      {len(X_test):,} amostras")

-> Removendo valores ausentes...
-> Aplicando rotina de limpeza no texto...
Total de registros válidos pós-limpeza: 853,982
-> Realizando divisão estratificada (70/15/15)...
Conjunto de Treino:     597,787 amostras
Conjunto de Validação:  128,097 amostras
Conjunto de Teste:      128,098 amostras


**3. Baseline 1 (TF-IDF + Regressão Logística)**




In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# 1. Vetorização por TF-IDF (Unigramas + Bigramas limitados a 5.000 termos)
print("-> Ajustando o vetorizador TF-IDF no conjunto de treino...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

# Transforma os conjuntos de dados em matrizes esparsas
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 2. Inicialização e treinamento da Regressão Logística com penalização L2
print("-> Treinando o classificador de Regressão Logística...")
modelo_lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
modelo_lr.fit(X_train_vec, y_train)

# 3. Inferência sobre o conjunto de teste
print("-> Gerando predições e probabilidades para o teste...")
y_pred = modelo_lr.predict(X_test_vec)
y_prob = modelo_lr.predict_proba(X_test_vec)[:, 1]

# 4. Exibição das métricas consolidadas
print("\n" + "="*45)
print("   RESULTADOS DO BASELINE 1 (TF-IDF + LR)   ")
print("="*45)
print(f"Acurácia:        {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score Macro:  {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"ROC-AUC:         {roc_auc_score(y_test, y_prob):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred, target_names=['Não Recomenda (0)', 'Recomenda (1)']))

-> Ajustando o vetorizador TF-IDF no conjunto de treino...
-> Treinando o classificador de Regressão Logística...
-> Gerando predições e probabilidades para o teste...

   RESULTADOS DO BASELINE 1 (TF-IDF + LR)   
Acurácia:        0.9622
F1-Score Macro:  0.7923
ROC-AUC:         0.9559

Relatório de Classificação Detalhado:
                   precision    recall  f1-score   support

Não Recomenda (0)       0.78      0.49      0.60      7503
    Recomenda (1)       0.97      0.99      0.98    120595

         accuracy                           0.96    128098
        macro avg       0.87      0.74      0.79    128098
     weighted avg       0.96      0.96      0.96    128098



**4. Download do GloVe, Tokenização e Matriz de Embeddings**

In [4]:
import os
import zipfile
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Download dos embeddings GloVe (NILC)
glove_txt = "glove_s100.txt"

if not os.path.exists(glove_txt):
    print("-> Baixando GloVe 100d...")
    !wget -q --show-progress "https://huggingface.co/datasets/liaad/glove-pt-br/resolve/main/glove_s100.txt" -O glove_s100.txt || true
    if not os.path.exists(glove_txt):
        !wget -q --show-progress "https://object.c3s.uni-duesseldorf.de/nilc/glove_s100.zip" -O glove_s100.zip && unzip -q glove_s100.zip || true

# 2. Tokenizacao e sequenciamento (T = 120)
MAX_LEN = 120
MAX_WORDS = 50_000
EMBEDDING_DIM = 100

print("-> Ajustando tokenizer nos dados de treino...")
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding='post', truncating='post')

# 3. Construcao da matriz de pesos
print("-> Montando matriz de embeddings...")
embeddings_index = {}
with open(glove_txt, encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

num_words_matrix = min(MAX_WORDS, len(tokenizer.word_index) + 1)
embedding_matrix = np.zeros((num_words_matrix, EMBEDDING_DIM))

for word, i in tokenizer.word_index.items():
    if i < MAX_WORDS:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

print(f"-> Matriz de embeddings construida: {embedding_matrix.shape}")

-> Baixando GloVe 100d...
-> Ajustando tokenizer nos dados de treino...
-> Montando matriz de embeddings...
-> Matriz de embeddings construida: (50000, 100)


**5.1. BiLSTM (Embeddings Congelados)**

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# 1. Arquitetura da rede BiLSTM
modelo_bilstm = Sequential([
    Embedding(
        input_dim=num_words_matrix,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

modelo_bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_bilstm.summary()

# 2. Early stopping monitorando a perda de validacao
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True,
    verbose=1
)

# 3. Treinamento
print("\n-> Treinando BiLSTM...")
history = modelo_bilstm.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=512,
    callbacks=[early_stop]
)

# 4. Avaliacao no conjunto de teste
y_prob_bilstm = modelo_bilstm.predict(X_test_seq, batch_size=512).ravel()
y_pred_bilstm = (y_prob_bilstm >= 0.5).astype(int)

print(f"\nAcuracia: {accuracy_score(y_test, y_pred_bilstm):.4f}")
print(f"F1 Macro: {f1_score(y_test, y_pred_bilstm, average='macro'):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_bilstm):.4f}\n")
print(classification_report(y_test, y_pred_bilstm, target_names=['Nao Recomenda', 'Recomenda']))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     5,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,000,000 (19.07 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 5,000,000 (19.07 MB)


-> Treinando BiLSTM...
Epoch 1/10
1168/1168 ━━━━━━━━━━━━━━━━━━━━ 77s 59ms/step - accuracy: 0.9406 - loss: 0.2284 - val_accuracy: 0.9414 - val_loss: 0.2230
Epoch 2/10
1168/1168 ━━━━━━━━━━━━━━━━━━━━ 72s 61ms/step - accuracy: 0.9414 - loss: 0.2241 - val_accuracy: 0.9414 - val_loss: 0.2233
Epoch 3/10
1168/1168 ━━━━━━━━━━━━━━━━━━━━ 72s 62ms/step - accuracy: 0.9414 - loss: 0.2239 - val_accuracy: 0.9414 - val_loss: 0.2232
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.
251/251 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step

Acuracia: 0.9414
F1 Macro: 0.4849
ROC-AUC:  0.5000

               precision    recall  f1-score   support

Nao Recomenda       0.00      0.00      0.00      7503
    Recomenda       0.94      1.00      0.97    120595

     accuracy                           0.94    128098
    macro avg       0.47      0.50      0.48    128098
 weighted avg       0.89      0.94      0.91    128098



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**5.2. BiLSTM com Pesos de Classe (Class Weights)**


In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# 1. Calculo automatico dos pesos para balancear as classes
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = {i: peso for i, peso in enumerate(pesos)}
print(f"-> Pesos aplicados: {class_weights_dict}")

# 2. Arquitetura da rede BiLSTM
modelo_bilstm = Sequential([
    Embedding(
        input_dim=num_words_matrix,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=False
    ),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(32)),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

modelo_bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 3. Callbacks para convergencia estavel
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5, verbose=1)
]

# 4. Treinamento com pesos de classe
print("\n-> Treinando BiLSTM com pesos balanceados...")
history = modelo_bilstm.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=256,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

# 5. Avaliacao no conjunto de teste
y_prob_bilstm = modelo_bilstm.predict(X_test_seq, batch_size=512).ravel()
y_pred_bilstm = (y_prob_bilstm >= 0.5).astype(int)

print(f"\nAcuracia: {accuracy_score(y_test, y_pred_bilstm):.4f}")
print(f"F1 Macro: {f1_score(y_test, y_pred_bilstm, average='macro'):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_bilstm):.4f}\n")
print(classification_report(y_test, y_pred_bilstm, target_names=['Nao Recomenda', 'Recomenda']))

-> Pesos aplicados: {0: np.float64(8.536155933171498), 1: np.float64(0.5311094013206059)}

-> Treinando BiLSTM com pesos balanceados...
Epoch 1/10
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 87s 36ms/step - accuracy: 0.5292 - loss: 0.6932 - val_accuracy: 0.9414 - val_loss: 0.6928 - learning_rate: 0.0010
Epoch 2/10
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6336 - loss: 0.6914
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 82s 35ms/step - accuracy: 0.5635 - loss: 0.6932 - val_accuracy: 0.0586 - val_loss: 0.6970 - learning_rate: 0.0010
Epoch 3/10
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 82s 35ms/step - accuracy: 0.4318 - loss: 0.6932 - val_accuracy: 0.9414 - val_loss: 0.6925 - learning_rate: 5.0000e-04
Epoch 4/10
2335/2336 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7727 - loss: 0.6907
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 82s 35ms/step - accuracy: 0.5333 - loss: 0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**5.3. BiLSTM com Fine-Tuning de Embeddings e Otimização**

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# 1. Calculo dos pesos para balanceamento
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = {i: peso for i, peso in enumerate(pesos)}

# 2. Arquitetura otimizada com fine-tuning de embeddings
modelo_bilstm = Sequential([
    Embedding(
        input_dim=num_words_matrix,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=True
    ),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

modelo_bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_bilstm.summary()

# 3. Callback de parada antecipada
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True,
    verbose=1
)

# 4. Treinamento
print("\n-> Treinando BiLSTM com fine-tuning...")
history = modelo_bilstm.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=6,
    batch_size=256,
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

# 5. Avaliacao no teste
y_prob_bilstm = modelo_bilstm.predict(X_test_seq, batch_size=512).ravel()
y_pred_bilstm = (y_prob_bilstm >= 0.5).astype(int)

print(f"\nAcuracia: {accuracy_score(y_test, y_pred_bilstm):.4f}")
print(f"F1 Macro: {f1_score(y_test, y_pred_bilstm, average='macro'):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_bilstm):.4f}\n")
print(classification_report(y_test, y_pred_bilstm, target_names=['Nao Recomenda', 'Recomenda']))

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │     5,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,000,000 (19.07 MB)

 Trainable params: 5,000,000 (19.07 MB)

 Non-trainable params: 0 (0.00 B)


-> Treinando BiLSTM com fine-tuning...
Epoch 1/6
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 62s 25ms/step - accuracy: 0.6211 - loss: 0.6932 - val_accuracy: 0.9414 - val_loss: 0.6927
Epoch 2/6
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 70s 30ms/step - accuracy: 0.4762 - loss: 0.6932 - val_accuracy: 0.0586 - val_loss: 0.6933
Epoch 3/6
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 73s 26ms/step - accuracy: 0.5252 - loss: 0.6932 - val_accuracy: 0.9414 - val_loss: 0.6919
Epoch 4/6
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 58s 25ms/step - accuracy: 0.5948 - loss: 0.6932 - val_accuracy: 0.0586 - val_loss: 0.6967
Epoch 5/6
2336/2336 ━━━━━━━━━━━━━━━━━━━━ 59s 25ms/step - accuracy: 0.3944 - loss: 0.6932 - val_accuracy: 0.0586 - val_loss: 0.6931
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 3.
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step

Acuracia: 0.9414
F1 Macro: 0.4849
ROC-AUC:  0.5000

               precision    recall  f1-score   support

Nao Recomenda       0.00      0.00      0.00      7503
    Recomenda   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
